In [ ]:
# bootstrap: Colab clone + local import of `gpurt` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_repo)], check=True)
    os.chdir(_repo / "02-cuda-nccl-runtime/cuda-and-nccl/cuda-nccl-lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "gpurt").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 02 · Memory-bound kernels on a real GPU: effective bandwidth, launch overhead and CUDA Graphs

**Tier:** T1. It needs any NVIDIA GPU (Colab or Kaggle T4, an L4, a rented RTX 4090) with `numba-cuda`
installed. If there is no GPU, or if Numba cannot use the GPU (no numba-cuda, a driver/toolkit
mismatch), the notebook takes its **T0 path**. `gpurt.kernels` detects the cause and tells you why.

On the T0 path, the same kernels run at small sizes in the simulator (correctness only). The notebook labels each
printed number *model prediction*. It calculates these numbers from byte counts and stated assumptions,
and never shows them as a measurement.

## The one-minute version

* You measure a memory-bound kernel in **bytes**: effective bandwidth = bytes the algorithm must move ÷
  time. Compare it with the datasheet peak. At 80–90 % of peak, the kernel is done. The only way to make it
  faster is to move fewer bytes (fusion, lower precision).
* Measure the kernel, not the infrastructure code: data already on the device, CUDA events, a warm-up launch.
* Small launches are **latency-bound**: $t = \alpha + S/B$, where $\alpha$ is a few microseconds of launch
  overhead. Notebook 04 fits the same model to collectives.
* Access patterns and extra passes cost bandwidth. Naive transpose ≪ tiled ≈ copy. Fused softmax
  moves half the bytes of the unfused softmax.
* A step with many small kernels is **launch-bound**: the CPU cannot issue kernels as fast as the GPU
  finishes them. CUDA Graphs replay the full sequence with one launch.

Concepts: [the primer](../../PRIMER.md) §2 *The execution model*, §3 *Memory access patterns*,
§4 *Streams, launch overhead and CUDA Graphs*. The roofline itself is layer 01.

In [ ]:
import sys

import numpy as np

from gpurt import env

from gpurt.kernels import SIMULATOR, cuda  # noqa: E402
from gpurt.kernels import bench, traffic  # noqa: E402
from gpurt.kernels import elementwise as ew  # noqa: E402
from gpurt.kernels import softmax as sm  # noqa: E402
from gpurt.kernels import transpose as tr  # noqa: E402
from gpurt.launch import LaunchModel  # noqa: E402

sys.setswitchinterval(1e-4)  # only matters for the simulator path
HAVE_GPU = (not SIMULATOR) and cuda.is_available()
KERNELS_RUN = SIMULATOR or HAVE_GPU  # False only if NUMBA_ENABLE_CUDASIM=0 was forced on an unusable GPU
print(env.describe())
if HAVE_GPU:
    DEV = bench.device_info()
    SPEC = traffic.spec_for(DEV["name"])
    if SPEC is None:
        SPEC = traffic.GPUS["T4"]
        print(f"{DEV['name']} is not in traffic.GPUS: fractions of peak below use a T4's {SPEC.mem_gbps} GB/s — ignore them")
    print(f"T1 path: measuring on {DEV['name']} (cc {DEV['cc']}, {DEV['sms']} SMs); datasheet spec used: {SPEC.name}, "
          f"{SPEC.mem_gbps} GB/s (verify)")
else:
    SPEC = traffic.GPUS["T4"]
    why = env.NUMBA_FALLBACK or "no GPU visible"
    print(f"T0 path ({why}).\nKernels run in the simulator for correctness; numbers below are MODEL PREDICTIONS "
          f"for a {SPEC.name} ({SPEC.mem_gbps} GB/s peak, verify), not measurements.")
    if not KERNELS_RUN:
        print("NUMBA_ENABLE_CUDASIM=0 is set but Numba cannot use the GPU: kernel cells are skipped. Unset it "
              "(gpurt then picks the simulator) or install numba-cuda, and restart the kernel.")
    print("To measure: run this notebook on a GPU, or `python -m gpurt.kernels.bench` (see deploy/any-gpu).")

## Bytes first

Before you measure a time, count the bytes that each kernel *must* move (`gpurt.kernels.traffic`):

In [ ]:
n = 1 << 24
rows = [("copy", traffic.elementwise_bytes("copy", n)), ("vec_add", traffic.elementwise_bytes("vec_add", n)),
        ("reduction", traffic.reduction_bytes(n)), ("transpose 4096^2", traffic.transpose_bytes(4096, 4096)),
        ("softmax unfused 4096^2", traffic.softmax_bytes(4096, 4096, "unfused")),
        ("softmax fused 4096^2", traffic.softmax_bytes(4096, 4096, "fused"))]
for name, b in rows:
    print(f"{name:>24}: {b / 2**20:8.0f} MiB   ideal time at {SPEC.mem_gbps} GB/s: {b / (SPEC.mem_gbps * 1e9) * 1e3:6.3f} ms")

## Exercise 2.1 — effective bandwidth

Write `effective(n_elements, bytes_per_element, seconds, peak_gbps)`. It returns `(gbps, fraction_of_peak)`
(GB = 1e9 bytes). Use it on a hypothetical run. The run is a `vec_add` of 2²⁴ floats (12 bytes per
element) that takes 0.70 ms on a T4 (320 GB/s).

In [ ]:
def effective(n_elements: int, bytes_per_element: int, seconds: float, peak_gbps: float) -> tuple[float, float]:
    gbps = n_elements * bytes_per_element / seconds / 1e9
    return gbps, gbps / peak_gbps

In [ ]:
gbps, frac = effective(1 << 24, 12, 0.70e-3, 320)
assert abs(gbps - 287.6) < 0.1 and abs(frac - 0.899) < 0.001
print(f"✅ {gbps:.1f} GB/s = {frac:.0%} of peak: a streaming kernel at ~90 % has nothing left to gain but fewer bytes")

## A bandwidth sweep: two regimes

Measure the time of `copy` and `vec_add` on vectors of 2¹⁰ to 2²⁶ floats (4 KiB to 256 MiB each). The
traffic is 8 KiB–512 MiB for copy and 12 KiB–768 MiB for vec_add. Small launches cost an approximately
constant $\alpha$ (launch + latency). Large launches approach the DRAM bandwidth $B$.

On the T0 path, we make sure in the simulator that the results are correct. We also print what the α-β model *predicts* for
the reference GPU. If you brought back `out/kernels.json` from a GPU box (`deploy/any-gpu`), we also print
the measurement from that box.

In [ ]:
from gpurt.dist.alphabeta import AlphaBeta, fit  # noqa: E402

if HAVE_GPU:
    sweep = bench.bandwidth_sweep("copy") + bench.bandwidth_sweep("vec_add")
    for r in sweep:
        print(f"{r['kind']:>8} n={r['n']:>10}  {r['seconds'] * 1e6:9.1f} µs  {r['gbps']:7.1f} GB/s  (measured)")
    MODEL = fit([r["bytes"] for r in sweep if r["kind"] == "copy"], [r["seconds"] for r in sweep if r["kind"] == "copy"])
    print("copy fit (measured):", MODEL)
else:
    if KERNELS_RUN:
        x = np.arange(1000, dtype=np.float32)
        assert np.array_equal(ew.run_copy(x, blocks=2, threads=64), x)
        assert np.array_equal(ew.run_vec_add(x, x), 2 * x)
        print("simulator: copy and vec_add are correct.")
    MODEL = AlphaBeta(alpha_s=5e-6, bw_Bps=SPEC.mem_gbps * 1e9 * 0.8)  # ASSUMPTIONS: 5 µs, 80 % of peak
    print("MODEL PREDICTION (α = 5 µs, B = 80 % of peak — assumptions):")
    for k in range(10, 27, 2):
        b = traffic.elementwise_bytes("copy", 2 ** k)
        print(f"    copy n=2^{k:<2}  {MODEL.time(b) * 1e6:9.1f} µs  {MODEL.algbw_gbps(b):7.1f} GB/s  ({MODEL.regime(b)})")
    import json  # noqa: E402
    from pathlib import Path  # noqa: E402

    for path in sorted(Path("..").glob("out/*kernels*.json")):  # results you brought back from a GPU box
        r = json.loads(path.read_text())
        print(f"{path.name}: MEASURED on {r['device']['name']}: copy α = {r['copy_alpha_us']:.1f} µs, "
              f"B = {r['copy_bandwidth_gbps']:.0f} GB/s; launch overhead {r['launch_overhead_us']:.1f} µs")

## Exercise 2.2 — where launch overhead stops mattering

The two terms of $t = \alpha + S/B$ are equal at $S_{1/2} = \alpha \cdot B$. At that size, the kernel gets
half of its asymptotic bandwidth. Write `crossover_elements(alpha_s, bw_Bps, bytes_per_element)`. It
returns the vector length at which a kernel that moves `bytes_per_element` bytes per element gets to that
point. Calculate it for a copy-like kernel with $\alpha$ = 5 µs and $B$ = 256 GB/s (80 % of a T4).

In [ ]:
def crossover_elements(alpha_s: float, bw_Bps: float, bytes_per_element: int) -> float:
    return alpha_s * bw_Bps / bytes_per_element

In [ ]:
assert abs(crossover_elements(5e-6, 256e9, 8) - 160_000) < 1e-6
mine = crossover_elements(MODEL.alpha_s, MODEL.bw_Bps, 8)
print(f"✅ below ~{mine:,.0f} floats a copy is launch-bound on {'this GPU' if HAVE_GPU else 'the model GPU'}: "
      "many small kernels waste the machine; fuse or batch them")

## Transpose: the cost of strided stores

Notebook 01 counted it: the stores of the naive transpose touch 32 sectors per warp request, not 4.
On a GPU, the effect is smaller than 8×, because L2 merges some partial sectors before they get to DRAM.
This is why you measure.

In [ ]:
if HAVE_GPU:
    tb = bench.transpose_bench(4096)
    for name, r in tb.items():
        print(f"{name:>13}: {r['gbps']:7.1f} GB/s  (measured)")
    COPY_GBPS = tb["copy2d"]["gbps"]
else:
    if KERNELS_RUN:
        X = np.random.default_rng(0).random((40, 70), dtype=np.float32)
        assert np.array_equal(tr.run_transpose(X, "naive"), X.T) and np.array_equal(tr.run_transpose(X, "tiled"), X.T)
        print("simulator: both transposes are correct.")
    COPY_GBPS = MODEL.bw_Bps / 1e9
    print(f"Model copy bandwidth: {COPY_GBPS:.0f} GB/s (assumption)")

## Exercise 2.3 — a no-cache model of the naive transpose

Half of the bytes are loads at full efficiency. The other half are stores at efficiency $e$ (useful bytes
÷ bytes moved, 1/8 for the naive transpose). If the hardware moves bytes at the copy bandwidth $B$, what
effective bandwidth does the transpose report? Write `naive_transpose_gbps(copy_gbps, store_efficiency)`.

In [ ]:
def naive_transpose_gbps(copy_gbps: float, store_efficiency: float) -> float:
    # bytes/2 at B, plus bytes/2 at B*e  ->  t = bytes/(2B) * (1 + 1/e)
    return 2 * copy_gbps / (1 + 1 / store_efficiency)

In [ ]:
assert abs(naive_transpose_gbps(300, 0.125) - 66.667) < 1e-3
assert naive_transpose_gbps(300, 1.0) == 300
pred = naive_transpose_gbps(COPY_GBPS, 0.125)
if HAVE_GPU:
    print(f"✅ model {pred:.0f} GB/s vs measured naive {tb['naive']['gbps']:.0f} GB/s — above the model means caches "
          "absorbed part of the waste; the tiled kernel should sit near copy")
else:
    print(f"✅ model prediction for the naive transpose: {pred:.0f} GB/s (22 % of copy) — a pessimistic bound; "
          "measure it on a GPU")

## Fusion: softmax in one pass instead of four kernels

The unfused softmax moves the matrix six times (24 B/element) in four launches. The fused *online*
softmax moves it three times (12 B/element) in one launch. In the terms of primer §3.5, this unfused path
already merges subtract and exp (6RC, not the primer's five-kernel 8RC). The fused kernel is the primer's
*online* row (3RC). It is not the 2RC variant that reads a row one time. The Triton cell in the next
subsection is that variant.

In [ ]:
if HAVE_GPU:
    sb = bench.softmax_bench(4096, 4096)
    for name, r in sb.items():
        print(f"{name:>8}: {r['seconds'] * 1e3:7.3f} ms  {r['gbps']:7.1f} GB/s effective  (measured)")
    print(f"speedup {sb['unfused']['seconds'] / sb['fused']['seconds']:.2f}x")
elif KERNELS_RUN:
    S = np.random.default_rng(1).standard_normal((6, 50)).astype(np.float32) * 3
    for fused in (True, False):
        np.testing.assert_allclose(sm.run_softmax(S, fused, threads=16), sm.softmax_reference(S), rtol=1e-5, atol=1e-6)
    print("simulator: fused and unfused softmax match NumPy")

### The same fusion in Triton (optional, T1)

`gpurt.kernels.triton_kernels` writes the softmax at the *block* level, in the same way as `torch.compile`.
It uses one program per row and holds the row in registers. It does one read and one write per element
(8 B/element in float32, the primer's 2RC row). With torch and triton on a GPU, this cell
compares the Triton softmax with `torch.softmax`. The cell also measures the time of the Triton softmax
next to the Numba kernels of the cells before.

In [ ]:
from gpurt.kernels import triton_kernels  # noqa: E402

if HAVE_GPU and triton_kernels.available():
    import torch  # noqa: E402

    xt = torch.randn(4096, 4096, device="cuda")
    torch.testing.assert_close(triton_kernels.softmax(xt), torch.softmax(xt, dim=1))
    torch.testing.assert_close(triton_kernels.vec_add(xt[0], xt[1]), xt[0] + xt[1])
    start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    triton_kernels.softmax(xt)  # warm-up (compiles)
    start.record()
    for _ in range(20):
        triton_kernels.softmax(xt)
    end.record()
    torch.cuda.synchronize()
    t = start.elapsed_time(end) / 1e3 / 20
    print(f"Triton softmax 4096^2: {t * 1e3:.3f} ms, {8 * xt.numel() / t / 1e9:.0f} GB/s effective (measured; "
          f"Numba fused above: {sb['fused']['seconds'] * 1e3:.3f} ms)")
else:
    print("Triton path skipped: it needs a GPU with torch and triton (pip install -e '.[torch,triton]'). It would check "
          "triton_kernels.softmax against torch.softmax and time it next to the Numba fused kernel.")

## Exercise 2.4 — predict the fusion speedup, including launches

Write `softmax_times(rows, cols, bw_Bps, launch_s)`. It returns `(t_unfused, t_fused)`: 24 B/element and
four launches, against 12 B/element and one launch. Compare a large matrix with a small one.

In [ ]:
def softmax_times(rows: int, cols: int, bw_Bps: float, launch_s: float) -> tuple[float, float]:
    elems = rows * cols
    return 4 * launch_s + 24 * elems / bw_Bps, launch_s + 12 * elems / bw_Bps

In [ ]:
tu, tf = softmax_times(4096, 4096, 300e9, 5e-6)
assert abs(tu / tf - 2.01) < 0.01
tu_small, tf_small = softmax_times(16, 128, 300e9, 5e-6)
assert 3.9 < tu_small / tf_small < 4.0
print(f"✅ large: {tu / tf:.2f}x (bytes decide); tiny: {tu_small / tf_small:.2f}x (launches decide)")

## Determinism: atomics change the order of additions

Floating-point addition is not associative. The atomic reduction adds the block results in the order in
which the blocks finish. Thus repeated runs can be different in the last bits. The two-pass reduction
always uses the same order.

In [ ]:
if HAVE_GPU:
    rb = bench.reduction_bench()
    print("distinct atomic results  :", rb["distinct_atomic_results"], "(measured, 5 runs)")
    print("distinct two-pass results:", rb["distinct_two_pass_results"])
    print("float64 reference        :", rb["float64_reference"])
else:
    v = np.random.default_rng(2).random(1 << 20, dtype=np.float32)
    orders = {float(np.add.reduce(v[p])) for p in (np.arange(v.size), np.random.default_rng(3).permutation(v.size),
                                                      np.random.default_rng(4).permutation(v.size))}
    print("NumPy, same numbers summed in three orders (float32):", sorted(orders))

## Launch overhead and CUDA Graphs

`gpurt.launch.LaunchModel`: eager step $\approx n \cdot \max(k + g, L)$, graph step $\approx G + n \cdot (k + g)$.
Here $n$ is the number of kernels of $k$ µs, and $g$ is a GPU-side gap. $L$ is the CPU launch cost, and
$G$ is one graph launch. On a GPU with torch, `measure_graph_vs_eager()` measures the real values.

In [ ]:
from gpurt.launch import measure_graph_vs_eager  # noqa: E402

if HAVE_GPU:
    print(f"Numba empty-kernel launch: {bench.launch_overhead(500):.1f} µs per launch (measured, host clock)")
    if env.has_module("torch"):
        g = measure_graph_vs_eager(n_kernels=200)
        print(f"torch, 200 tiny kernels: eager {g['eager_us']:.0f} µs, graph {g['graph_us']:.0f} µs, "
              f"{g['speedup']:.1f}x (measured; replay correct: {g['replay_correct']})")
else:
    m = LaunchModel()  # primer §4.2's assumptions
    print(f"MODEL (assumptions: L = {m.launch_us} µs, G = {m.graph_launch_us} µs, g = {m.node_gap_us} µs):")
    for k in (2, 5, 20):
        print(f"    200 kernels of {k:>2} µs: eager {m.eager_us(200, k):6.0f} µs, graph {m.graph_us(200, k):6.0f} µs, "
              f"launch-bound: {m.launch_bound(k)}")

## Exercise 2.5 — should this decode step be captured in a graph?

A decode step runs `layers × kernels_per_layer` kernels. Write `decode_step(layers, kernels_per_layer,
kernel_us, model)` with a `LaunchModel`. It returns `(eager_us, graph_us, launch_bound)`.

Calculate the scenario of primer §4.2. It has 32 layers × 12 kernels = 384 kernels, ≈2 µs each at batch 1
and ≈20 µs at a large batch. Use its assumptions: $L$ = 5 µs, $G$ = 10 µs and no GPU-side gap. Then add the gap that the
primer leaves out: $g$ = 1 µs between consecutive kernels.

In [ ]:
def decode_step(layers: int, kernels_per_layer: int, kernel_us: float, model: LaunchModel):
    n = layers * kernels_per_layer
    return model.eager_us(n, kernel_us), model.graph_us(n, kernel_us), model.launch_bound(kernel_us)

In [ ]:
primer = LaunchModel(launch_us=5.0, graph_launch_us=10.0, node_gap_us=0.0)
eager, graph, bound = decode_step(32, 12, 2.0, primer)
assert (eager, graph, bound) == (1920.0, 778.0, True)  # primer §4.2: 1,922 vs 778 µs (it adds the last kernel)
eager_big, graph_big, bound_big = decode_step(32, 12, 20.0, primer)
assert (eager_big, graph_big, bound_big) == (7680.0, 7690.0, False)
eager_gap, graph_gap, _ = decode_step(32, 12, 2.0, LaunchModel(launch_us=5.0, graph_launch_us=10.0, node_gap_us=1.0))
assert (eager_gap, graph_gap) == (1920.0, 1162.0)
print(f"✅ batch 1: {eager / graph:.2f}x faster as a graph, {eager_gap / graph_gap:.2f}x once each kernel also costs "
      "a 1 µs GPU-side gap; large batch: no gain — graphs remove CPU cost, not GPU work. Engines capture decode "
      "graphs per batch size, not prefill.")

## In a design review

**Two minutes.** For memory-bound kernels, I report *effective bandwidth* next to the datasheet peak.
Effective bandwidth is the bytes that the algorithm must move, divided by the time. I measure with data
on the device and CUDA events around many back-to-back launches, after a warm-up. A size sweep shows two
regimes: a latency floor of a few microseconds per launch, and a bandwidth plateau. The α-β fit gives the
two numbers and the size where they cross.

A kernel near 80–90 % of peak is finished. Below that, look at access patterns (the strided stores of the
naive transpose) or extra passes (unfused softmax moves two times the bytes). When a step is hundreds of
small kernels, the CPU is the bottleneck, and `GPU util` still shows a high value. Then CUDA Graphs, or
fewer, fused kernels, are the solution.

**Drill questions**

1. *The first call of my kernel takes 300 ms, and the other calls take 50 µs. Is the GPU slow?* No. The
   first call does a JIT compilation (Numba, Triton, PTX JIT), and it can also create the CUDA context. Do
   a warm-up before you measure the time. In production, compile in advance or cache the result (engines do
   a warm-up and capture graphs at start-up).
2. *Our fused kernel gets 88 % of peak DRAM bandwidth. How do we make it faster?* Move fewer bytes: use
   lower precision (bf16/fp8), fuse the adjacent op, or do not read the inputs again. More threads do not
   help a saturated memory system.
3. *When do CUDA Graphs not help?* They do not help when kernels are sufficiently long that the CPU stays
   ahead (large batches, prefill). They also do not help when shapes change at each step. An engine
   captures one graph per shape. Thus engines pad to a set of captured batch sizes.